In [2]:
import h5py
import networkx as nx
import numpy as np
from collections import defaultdict

file_path = "./SMA_data_processing/cobre_combined_connectomes_database.h5"

graphs_hc_p = []
graphs_scz_p = []

graphs_hc_g = []
graphs_scz_g = []

with h5py.File(file_path, "r") as f:
    hc_p = f["hc_pearson"]
    scz_p = f["scz_pearson"]

    hc_g = f["hc_glasso"]
    scz_g = f["scz_glasso"]

    for i in range(hc_p.shape[0]):
        A = hc_p[i]
        A = A.copy()
        np.fill_diagonal(A, 0)
        graphs_hc_p.append(nx.from_numpy_array(A))

    for i in range(scz_p.shape[0]):
        A = scz_p[i]
        A = A.copy()
        np.fill_diagonal(A, 0)
        graphs_scz_p.append(nx.from_numpy_array(A))

    for i in range(hc_g.shape[0]):
        A = hc_g[i]
        A = A.copy()
        np.fill_diagonal(A, 0)
        graphs_hc_g.append(nx.from_numpy_array(A))

    for i in range(scz_g.shape[0]):
        A = scz_g[i]
        A = A.copy()
        np.fill_diagonal(A, 0)
        graphs_scz_g.append(nx.from_numpy_array(A))

print("HC graphs:", len(graphs_hc_p))
print("SCZ graphs:", len(graphs_scz_p))
print("HC graphs (Glasso):", len(graphs_hc_g))
print("SCZ graphs (Glasso):", len(graphs_scz_g))
print(graphs_hc_p[0])
print(list(graphs_hc_p[0].neighbors(0)))
print(graphs_hc_g[0])
print(list(graphs_hc_g[0].neighbors(0)))

G = graphs_hc_p[0]

HC graphs: 73
SCZ graphs: 72
HC graphs (Glasso): 73
SCZ graphs (Glasso): 72
Graph with 1019 nodes and 518671 edges
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 19

Reference : https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.community.louvain.louvain_communities.html

The algorithm works in 2 steps. On the first step it assigns every node to be in its own community and then for each node it tries to find the maximum positive modularity gain by moving each node to all of its neighbor communities. If no positive gain is achieved the node remains in its original community.

The modularity gain obtained by moving an **isolated node** into a community can easily be calculated by the following formula :

$$
\Delta Q = \frac{k_{i,in}}{2m} - \gamma \frac{\Sigma_{tot}\cdot k_i}{2m^2}
$$

where :

- $m$ is the size of the graph
- $k_{i,\mathrm{in}}$ is the sum of the weights of the links from $i$ to nodes in $C$
- $k_i$ is the sum of the weights of the links incident to node $i$
- $\Sigma_{\mathrm{tot}}$ is the sum of the weights of the links incident to nodes in $C$
- $\gamma$ is the resolution parameter


In [3]:
def initial_assignment_communities(G) :
    return {node: node for node in G.nodes()}

In [4]:
def delta_q(G, node, target_community, communities, gamma=1.0, weight="weight"):

    degrees = dict(G.degree(weight=weight))
    two_m = sum(degrees.values())

    k_i_in = 0.0

    for neighbor, edge_data in G[node].items():
        if neighbor != node and communities[neighbor] == target_community:
            k_i_in += edge_data.get(weight)

    k_i = degrees[node]

    sigma_tot = 0.0

    for n in G.nodes():
        if n != node and communities[n] == target_community:
            sigma_tot += degrees[n]

    return (k_i_in / two_m) - gamma * (sigma_tot * k_i) / (two_m * two_m)

In [5]:
def louvain_step_one(G, gamma=1.0, weight="weight"):
    
    communities = initial_assignment_communities(G)
    moved = True

    while moved:
        moved = False

        for node in G.nodes():
            current_community = communities[node]

            # change community to isolate node
            communities[node] = -1
            
            #unique values
            target_communities = set()

            for neighbor in G.neighbors(node):
                if communities[neighbor] != -1:
                    target_communities.add(communities[neighbor])

            best_community = current_community
            best_dq = 0.0

            for tc in target_communities:
                dq = delta_q(G, node, tc, communities, gamma=gamma, weight=weight)
                if dq > best_dq:
                    best_dq = dq
                    best_community = tc

            communities[node] = best_community

            if best_community != current_community:
                moved = True

    return communities

In [6]:
communities = louvain_step_one(G)
print(communities)

{0: 386, 1: 497, 2: 497, 3: 497, 4: 386, 5: 497, 6: 497, 7: 497, 8: 386, 9: 386, 10: 497, 11: 497, 12: 497, 13: 386, 14: 386, 15: 497, 16: 386, 17: 386, 18: 497, 19: 386, 20: 497, 21: 497, 22: 497, 23: 386, 24: 497, 25: 497, 26: 386, 27: 386, 28: 497, 29: 386, 30: 497, 31: 497, 32: 386, 33: 497, 34: 497, 35: 497, 36: 497, 37: 497, 38: 386, 39: 386, 40: 497, 41: 497, 42: 386, 43: 386, 44: 497, 45: 497, 46: 497, 47: 497, 48: 89, 49: 497, 50: 497, 51: 386, 52: 497, 53: 497, 54: 497, 55: 497, 56: 497, 57: 497, 58: 89, 59: 497, 60: 497, 61: 497, 62: 497, 63: 497, 64: 497, 65: 497, 66: 497, 67: 497, 68: 497, 69: 497, 70: 497, 71: 497, 72: 497, 73: 497, 74: 497, 75: 497, 76: 497, 77: 497, 78: 497, 79: 89, 80: 497, 81: 89, 82: 89, 83: 89, 84: 89, 85: 89, 86: 89, 87: 386, 88: 89, 89: 89, 90: 89, 91: 89, 92: 89, 93: 386, 94: 89, 95: 89, 96: 89, 97: 89, 98: 89, 99: 89, 100: 89, 101: 89, 102: 89, 103: 89, 104: 89, 105: 89, 106: 89, 107: 89, 108: 89, 109: 89, 110: 89, 111: 89, 112: 89, 113: 89, 114

The second phase consists in building a new network whose nodes are now the communities found in the first phase. To do so, the weights of the links between the new nodes are given by the sum of the weight of the links between nodes in the corresponding two communities. Once this phase is complete it is possible to reapply the first phase creating bigger communities with increased modularity.

In [7]:
def louvain_step_two(G, communities, weight="weight"):
    new_G = nx.Graph()

    unique_communities = set(communities.values())
    for uc in unique_communities:
        new_G.add_node(uc)

    for n1, n2, edge_data in G.edges(data=True):
        c_n1 = communities[n1]
        c_n2 = communities[n2]
        w = edge_data.get(weight)

        if new_G.has_edge(c_n1, c_n2):
            new_G[c_n1][c_n2][weight] += w
        else:
            new_G.add_edge(c_n1, c_n2, **{weight: w})

    return new_G

In [8]:
new_G = louvain_step_two(G, communities)
print("Original :", G)
print("New : ", new_G)
print(new_G.nodes())

Original : Graph with 1019 nodes and 518671 edges
New :  Graph with 3 nodes and 6 edges
[89, 497, 386]


The above two phases are executed until no modularity gain is achieved (or is less than the threshold, or until max_levels is reached).

$$
Q = \frac{1}{2m} \sum_{ij} \left( A_{ij} - \frac{k_i k_j}{2m} \right)\delta(c_i, c_j)
$$

In [ ]:
def modularity(G, communities, weight='weight'):
    
    degrees = dict(G.degree(weight=weight))
    two_m = sum(degrees.values())
    sigma = 0.0
    nodes = list(G.nodes())

    for i in nodes:
        for j in nodes:
            if communities[i] == communities[j]:
                if G.has_edge(i, j):
                    A_ij = G[i][j].get(weight, 1.0)
                else:
                    A_ij = 0
                sigma += A_ij - (degrees[i] * degrees[j]) / two_m

    return sigma / two_m

In [10]:
print(modularity(G, communities))

grouped = defaultdict(set)
for node, cid in communities.items():
    grouped[cid].add(node)

communities_nx = list(grouped.values())

nx.algorithms.community.quality.modularity(G, communities_nx, weight='weight', resolution=1)

print(communities)
print(communities_nx)


0.10025725476368706
{0: 386, 1: 497, 2: 497, 3: 497, 4: 386, 5: 497, 6: 497, 7: 497, 8: 386, 9: 386, 10: 497, 11: 497, 12: 497, 13: 386, 14: 386, 15: 497, 16: 386, 17: 386, 18: 497, 19: 386, 20: 497, 21: 497, 22: 497, 23: 386, 24: 497, 25: 497, 26: 386, 27: 386, 28: 497, 29: 386, 30: 497, 31: 497, 32: 386, 33: 497, 34: 497, 35: 497, 36: 497, 37: 497, 38: 386, 39: 386, 40: 497, 41: 497, 42: 386, 43: 386, 44: 497, 45: 497, 46: 497, 47: 497, 48: 89, 49: 497, 50: 497, 51: 386, 52: 497, 53: 497, 54: 497, 55: 497, 56: 497, 57: 497, 58: 89, 59: 497, 60: 497, 61: 497, 62: 497, 63: 497, 64: 497, 65: 497, 66: 497, 67: 497, 68: 497, 69: 497, 70: 497, 71: 497, 72: 497, 73: 497, 74: 497, 75: 497, 76: 497, 77: 497, 78: 497, 79: 89, 80: 497, 81: 89, 82: 89, 83: 89, 84: 89, 85: 89, 86: 89, 87: 386, 88: 89, 89: 89, 90: 89, 91: 89, 92: 89, 93: 386, 94: 89, 95: 89, 96: 89, 97: 89, 98: 89, 99: 89, 100: 89, 101: 89, 102: 89, 103: 89, 104: 89, 105: 89, 106: 89, 107: 89, 108: 89, 109: 89, 110: 89, 111: 89, 1

In [11]:
def louvain(G, weight="weight", threshold=1e-07, max_levels=1000, gamma=1.0):
    current_G = G
    levels = 0

    communities_original_graph = {node: node for node in G.nodes()}

    prev_modularity = -1

    while levels < max_levels:

        communities = louvain_step_one(current_G, gamma)

        curr_modularity = modularity(current_G, communities, weight=weight)

        if prev_modularity != -1 and abs(curr_modularity - prev_modularity) < threshold:
            break

        prev_modularity = curr_modularity

        new_G = louvain_step_two(current_G, communities, weight=weight)

        new_communities_original_graph = {}
        for original_node, old_comm in communities_original_graph.items():
            new_communities_original_graph[original_node] = communities[old_comm]

        communities_original_graph = new_communities_original_graph
        current_G = new_G
        levels += 1

    return communities_original_graph

In [12]:
com_nx = nx.algorithms.community.louvain_communities(G, weight='weight', resolution=1, threshold=1e-07, max_level=1000, seed=None)
q_nx = nx.algorithms.community.quality.modularity(G, com_nx, weight='weight', resolution=1)

print(q_nx)

com = louvain(G, weight="weight", threshold=1e-07, max_levels=1000, gamma=1.0)
q = modularity(G, com)

print(q)

0.09782877691397933
0.10025725476368706


In [ ]:
def delta_q(G, node, target_community, communities, degrees, two_m, comm_degrees, gamma=1.0, weight="weight"):
    k_i_in = 0.0
    for neighbor, edge_data in G[node].items():
        if neighbor != node and communities[neighbor] == target_community:
            k_i_in += edge_data.get(weight, 1.0)

    k_i = degrees[node]

    # Get sigma_tot with Dictionary lookup instead of the loop
    sigma_tot = comm_degrees.get(target_community, 0.0)

    return (k_i_in / two_m) - gamma * (sigma_tot * k_i) / (two_m * two_m)

In [ ]:
def louvain_step_one(G, gamma=1.0, weight="weight"):
    communities = initial_assignment_communities(G)

    # ── Pre-compute once (instead of every delta_q call) ──
    degrees = dict(G.degree(weight=weight))
    two_m = sum(degrees.values())

    # Build comm_degrees: community_id → sum of degrees of its members
    comm_degrees = {}
    for node, comm in communities.items():
        comm_degrees[comm] = comm_degrees.get(comm, 0.0) + degrees[node]

    moved = True

    while moved:
        moved = False

        for node in G.nodes():
            current_community = communities[node]
            k_i = degrees[node]

            # Remove node from its current community
            communities[node] = -1
            comm_degrees[current_community] -= k_i

            # Find neighbor communities
            target_communities = set()
            for neighbor in G.neighbors(node):
                if communities[neighbor] != -1:
                    target_communities.add(communities[neighbor])
            best_community = current_community
            best_dq = 0.0

            for tc in target_communities:
                dq = delta_q(G, node, tc, communities, degrees, two_m, comm_degrees, gamma=gamma, weight=weight)
                if dq > best_dq:
                    best_dq = dq
                    best_community = tc

            # Place node in best community and update comm_degrees
            communities[node] = best_community
            comm_degrees[best_community] = comm_degrees.get(best_community, 0.0) + k_i

            if best_community != current_community:
                moved = True

    return communities


In [18]:
q_hc_p_tot = 0
q_scz_p_tot = 0

n_hc_p = len(graphs_hc_p)
n_scz_p = len(graphs_scz_p)

for graph in graphs_hc_p :
    com = louvain(graph, weight="weight", threshold=1e-07, max_levels=1000, gamma=1.0)
    q = modularity(graph, com)
    q_hc_p_tot += q

for graph in graphs_scz_p :
    com = louvain(graph, weight="weight", threshold=1e-07, max_levels=1000, gamma=1.0)
    q = modularity(graph, com)
    q_scz_p_tot += q

print(f"Healthy controls, Pearson: {q_hc_p_tot/n_hc_p}")
print(f"Schizophrenia, Pearson: {q_scz_p_tot/n_scz_p}")

q_hc_g_tot = 0
q_scz_g_tot = 0

n_hc_g = len(graphs_hc_g)
n_scz_g = len(graphs_scz_g)

for graph in graphs_hc_g :
    com = louvain(graph, weight="weight", threshold=1e-07, max_levels=1000, gamma=1.0)
    q = modularity(graph, com)
    q_hc_g_tot += q

for graph in graphs_scz_g :
    com = louvain(graph, weight="weight", threshold=1e-07, max_levels=1000, gamma=1.0)
    q = modularity(graph, com)
    q_scz_g_tot += q

print(f"Healthy controls, Glasso: {q_hc_g_tot/n_hc_g}")
print(f"Schizophrenia, Glasso: {q_scz_g_tot/n_scz_g}")

Healthy controls, Pearson: 0.2957318395734126
Schizophrenia, Pearson: 0.4950992767577772
Healthy controls, Glasso: 0.7347091776568656
Schizophrenia, Glasso: 0.7434962026982309


In [19]:
q_hc_p_tot_nx = 0
q_scz_p_tot_nx = 0

n_hc_p_nx = len(graphs_hc_p)
n_scz_p_nx = len(graphs_scz_p)

for graph in graphs_hc_p :
    com_nx = nx.algorithms.community.louvain_communities(graph, weight='weight', resolution=1, threshold=1e-07, max_level=1000, seed=None)
    q_nx = nx.algorithms.community.quality.modularity(graph, com_nx, weight='weight', resolution=1)
    q_hc_p_tot_nx += q_nx

for graph in graphs_scz_p :
    com_nx = nx.algorithms.community.louvain_communities(graph, weight='weight', resolution=1, threshold=1e-07, max_level=1000, seed=None)
    q_nx = nx.algorithms.community.quality.modularity(graph, com_nx, weight='weight', resolution=1)
    q_scz_p_tot_nx += q_nx

print(f"Healthy Controls, Pearson: {q_hc_p_tot_nx}/{n_hc_p_nx}")
print(f"Schizophrenia, Pearson: {q_scz_p_tot_nx}/{n_scz_p_nx}")

q_hc_g_tot_nx = 0
q_scz_g_tot_nx = 0

n_hc_g_nx = len(graphs_hc_g)
n_scz_g_nx = len(graphs_scz_g)

for graph in graphs_hc_g :
    com_nx = nx.algorithms.community.louvain_communities(graph, weight='weight', resolution=1, threshold=1e-07, max_level=1000, seed=None)
    q_nx = nx.algorithms.community.quality.modularity(graph, com_nx, weight='weight', resolution=1)
    q_hc_g_tot_nx += q_nx

for graph in graphs_scz_g :
    com_nx = nx.algorithms.community.louvain_communities(graph, weight='weight', resolution=1, threshold=1e-07, max_level=1000, seed=None)
    q_nx = nx.algorithms.community.quality.modularity(graph, com_nx, weight='weight', resolution=1)
    q_scz_g_tot_nx += q_nx

print(f"Healthy Controls, Glasso: {q_hc_g_tot_nx}/{n_hc_g_nx}")
print(f"Schizophrenia, Glasso: {q_scz_g_tot_nx}/{n_scz_g_nx}")

Healthy Controls, Pearson: 21.55150040746066/73
Schizophrenia, Pearson: 35.65312206829809/72
Healthy Controls, Glasso: 53.72922977932308/73
Schizophrenia, Glasso: 53.58444094176625/72
